In [1]:
import http.client

conn = http.client.HTTPSConnection("api.collectapi.com")

headers = {
    'content-type': "application/json",
    'authorization': "apikey 7IVNn8p9MaSxTGXF9JoUo1:3TxTO2SWdXCXHuNaO89eoV"
    }


conn.request("GET", "/gasPrice/stateUsaPrice?state=WA", headers=headers)

res = conn.getresponse()
data = res.read()

In [2]:
print(data.decode('utf-8'))

{"success":true,"result":{"state":[{"currency":"usd","name":"Washington","lowername":"washington","gasoline":"5.1890","midGrade":"5.4760","premium":"5.7190","diesel":"6.0470"}],"cities":[{"currency":"usd","gasoline":"4.8870","midGrade":"5.0660","premium":"5.4390","diesel":"5.6000","name":"Bellingham","lowername":"bellingham"},{"currency":"usd","gasoline":"5.4070","midGrade":"5.6290","premium":"5.8460","diesel":"6.1300","name":"Bremerton","lowername":"bremerton"},{"currency":"usd","gasoline":"5.1680","midGrade":"5.3850","premium":"5.6290","diesel":"6.2110","name":"Olympia","lowername":"olympia"},{"currency":"usd","gasoline":"4.9250","midGrade":"5.2600","premium":"5.5070","diesel":"5.7490","name":"Richland-Kennewick-Pasco","lowername":"richland-kennewick-pasco"},{"currency":"usd","gasoline":"5.5080","midGrade":"5.7580","premium":"5.9920","diesel":"6.3400","name":"Seattle-Bellevue-Everett","lowername":"seattle-bellevue-everett"},{"currency":"usd","gasoline":"4.6390","midGrade":"4.8960","p

In [3]:
##this lines of code  are used to convert data from bytes to dictionary
import json
data1 = json.loads(data)
type(data1)

dict

In [6]:
##Extracting Cities data
import pandas as pd
cities = data1['result']['cities']
cities = pd.DataFrame(cities)
cities.head()

,currency,gasoline,midGrade,premium,diesel,name,lowername
0,usd,4.8870,5.0660,5.4390,5.6000,Bellingham,bellingham
1,usd,5.4070,5.6290,5.8460,6.1300,Bremerton,bremerton
2,usd,5.1680,5.3850,5.6290,6.2110,Olympia,olympia
3,usd,4.9250,5.2600,5.5070,5.7490,Richland-Kennewick-Pasco,richland-kennewick-pasco
4,usd,5.5080,5.7580,5.9920,6.3400,Seattle-Bellevue-Everett,seattle-bellevue-everett


In [7]:
## Dropping the lowername
cities.drop(columns='lowername', inplace=True)
cities.head()

,currency,gasoline,midGrade,premium,diesel,name
0,usd,4.8870,5.0660,5.4390,5.6000,Bellingham
1,usd,5.4070,5.6290,5.8460,6.1300,Bremerton
2,usd,5.1680,5.3850,5.6290,6.2110,Olympia
3,usd,4.9250,5.2600,5.5070,5.7490,Richland-Kennewick-Pasco
4,usd,5.5080,5.7580,5.9920,6.3400,Seattle-Bellevue-Everett


In [8]:
cities.rename(columns={'name':'city'})
cities.head()

,currency,gasoline,midGrade,premium,diesel,name
0,usd,4.8870,5.0660,5.4390,5.6000,Bellingham
1,usd,5.4070,5.6290,5.8460,6.1300,Bremerton
2,usd,5.1680,5.3850,5.6290,6.2110,Olympia
3,usd,4.9250,5.2600,5.5070,5.7490,Richland-Kennewick-Pasco
4,usd,5.5080,5.7580,5.9920,6.3400,Seattle-Bellevue-Everett


In [11]:
## Creating and testing the database
from sqlalchemy import create_engine, text
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_NAME = os.getenv('DATABASE_NAME')
DATABASE_USER = os.getenv('DATABASE_USER')
DATABASE_PORT = os.getenv('DATABASE_PORT')
DATABASE_HOST = os.getenv('DATABASE_HOST')
DATABASE_PASSWORD = os.getenv('DATABASE_PASSWORD')

engine = create_engine(f'postgresql+psycopg2://{DATABASE_USER}:{DATABASE_PASSWORD}@{DATABASE_HOST}:{DATABASE_PORT}/{DATABASE_NAME}')
with engine.connect() as conn:
    result = conn.execute(text('SELECT 1'))
    for i in result:
        print(i)

cities.to_sql('cities_db',engine, if_exists='replace', index=False)
print('Cities database has been created successfully')


(1,)
Cities database has been created successfully


The data has been successfully loaded onto the database, the next step will be factorizing the code after which we will modularize the code